<a href="https://colab.research.google.com/github/zly554411-arch/ECON3916-Statistical-Machine-Learning/blob/main/Lab%2013/Lab_13_The_Architecture_of_Dimensionality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/zly554411-arch/ECON3916-Statistical-Machine-Learning/refs/heads/main/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [2]:
naive_model = smf.ols("Sale_Price ~ Property_Age", data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        19:53:26   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [3]:
# Step 2: The Multivariate Model
multi_model = smf.ols("Sale_Price ~ Property_Age + Distance_to_Tech_Hub", data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        19:57:13   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [5]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid
df['Price_Residuals']

,Price_Residuals
0,-56823.332740
1,19000.990249
2,-69149.815200
3,18481.157582
4,-2619.815668
...,...
995,21560.763160
996,-1940.516556
997,-3398.817768
998,2026.560249


In [7]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid
df['Age_Residuals']

,Age_Residuals
0,0.621803
1,-13.689856
2,3.233510
3,5.347789
4,4.053610
...,...
995,-14.814575
996,-6.511286
997,-1.986001
998,-4.589856


In [8]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802138


In [9]:
"""
Multivariate OLS Regression with 3D Interactive Hyperplane Visualization
=========================================================================
Predicts Sale_Price from Property_Age and Distance_to_Tech_Hub using
statsmodels, then renders the fitted regression hyperplane in Plotly.
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── 0. Reproducible synthetic dataset ────────────────────────────────────────
np.random.seed(42)
n = 250

property_age        = np.random.uniform(1, 50, n)          # years
distance_to_hub     = np.random.uniform(0.5, 30, n)        # km

# True DGP: intercept=800k, age=-3k/yr, distance=-8k/km, Gaussian noise
sale_price = (
    800_000
    - 3_000  * property_age
    - 8_000  * distance_to_hub
    + np.random.normal(0, 30_000, n)
)

df = pd.DataFrame({
    "Property_Age":        property_age,
    "Distance_to_Tech_Hub": distance_to_hub,
    "Sale_Price":           sale_price,
})

# ── 1. OLS regression via statsmodels ────────────────────────────────────────
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])
y = df["Sale_Price"]

model  = sm.OLS(y, X).fit()
print(model.summary())

# ── 2. Extract coefficients from the fitted model ────────────────────────────
# model.params is a pandas Series indexed by the column names we passed to sm.
# Pulling each coefficient by name makes the code robust to column reordering.
intercept   = model.params["const"]                # β₀
coef_age    = model.params["Property_Age"]         # β₁  (∂Price / ∂Age)
coef_dist   = model.params["Distance_to_Tech_Hub"] # β₂  (∂Price / ∂Distance)

print(f"\nExtracted coefficients:")
print(f"  Intercept            : {intercept:,.2f}")
print(f"  β₁ (Property_Age)    : {coef_age:,.2f}")
print(f"  β₂ (Dist_to_Hub)     : {coef_dist:,.2f}")

# ── 3. Build the meshgrid for the regression hyperplane ───────────────────────
# np.linspace creates 60 evenly-spaced values across each predictor's observed
# range.  np.meshgrid then produces two 2-D arrays (age_grid, dist_grid) so
# that every (row, col) pair corresponds to one (age, distance) combination —
# i.e. the full Cartesian product of the two axes.
age_vals  = np.linspace(df["Property_Age"].min(),
                         df["Property_Age"].max(), 60)
dist_vals = np.linspace(df["Distance_to_Tech_Hub"].min(),
                         df["Distance_to_Tech_Hub"].max(), 60)

# age_grid and dist_grid are both (60 × 60) arrays.
# indexing='xy' means rows vary with dist_vals and columns vary with age_vals
# (Plotly's surface expects this orientation — rows → y-axis, cols → x-axis).
age_grid, dist_grid = np.meshgrid(age_vals, dist_vals, indexing="xy")

# Apply the OLS equation element-wise across the entire grid:
#   Ŷ = β₀ + β₁·Age + β₂·Distance
# Because age_grid and dist_grid are the same shape, this produces a (60×60)
# surface of predicted Sale_Prices — one prediction per grid point.
price_grid = intercept + coef_age * age_grid + coef_dist * dist_grid

# ── 4. Residuals for colour-coding the scatter ────────────────────────────────
y_hat     = model.fittedvalues
residuals = y - y_hat

# ── 5. Build the Plotly figure ────────────────────────────────────────────────
fig = go.Figure()

# — 5a. Regression hyperplane (surface) ———————————————————————————————————————
# x / y / z map to the three axes of the 3-D plot:
#   x → Property_Age  (60 evenly-spaced values)
#   y → Distance_to_Tech_Hub  (60 evenly-spaced values)
#   z → Predicted Sale_Price  (60 × 60 grid of OLS predictions)
fig.add_trace(go.Surface(
    x=age_vals,          # 1-D array → Plotly infers the column axis
    y=dist_vals,         # 1-D array → Plotly infers the row axis
    z=price_grid,        # 2-D array (rows=dist, cols=age) of Ŷ values
    colorscale="Blues",
    opacity=0.55,
    name="OLS Hyperplane",
    showscale=False,
    contours={
        "z": {"show": True, "color": "royalblue", "width": 1, "highlightwidth": 2}
    },
    hovertemplate=(
        "Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Predicted Price: $%{z:,.0f}<extra>Hyperplane</extra>"
    ),
))

# — 5b. Actual data points (scatter3d), coloured by residual ——————————————————
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    name="Observed Sales",
    marker=dict(
        size=4.5,
        color=residuals,                # residual magnitude drives colour
        colorscale="RdYlGn",           # red = under-predicted, green = over-predicted
        cmin=-residuals.abs().max(),
        cmax= residuals.abs().max(),
        colorbar=dict(
            title="Residual ($)",
            x=1.02,
            thickness=14,
            len=0.6,
        ),
        opacity=0.85,
        line=dict(width=0.4, color="white"),
    ),
    hovertemplate=(
        "<b>Observed</b><br>"
        "Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Sale Price: $%{z:,.0f}<br>"
        "<extra></extra>"
    ),
))

# — 5c. Residual drop-lines from each point to the hyperplane —————————————————
# Each pair of rows draws one vertical line: observed point → fitted point.
# Plotly renders disconnected segments when None is inserted between groups.
line_x, line_y, line_z = [], [], []
for i in range(len(df)):
    line_x += [df["Property_Age"].iloc[i],        df["Property_Age"].iloc[i],        None]
    line_y += [df["Distance_to_Tech_Hub"].iloc[i], df["Distance_to_Tech_Hub"].iloc[i], None]
    line_z += [df["Sale_Price"].iloc[i],           y_hat.iloc[i],                      None]

fig.add_trace(go.Scatter3d(
    x=line_x, y=line_y, z=line_z,
    mode="lines",
    name="Residuals",
    line=dict(color="rgba(180,180,180,0.35)", width=1),
    hoverinfo="skip",
))

# ── 6. Layout & annotation ────────────────────────────────────────────────────
r2   = model.rsquared
rmse = np.sqrt(np.mean(residuals**2))

fig.update_layout(
    title=dict(
        text=(
            f"<b>OLS Regression Hyperplane</b> — Sale Price = "
            f"{intercept:,.0f} "
            f"{'−' if coef_age < 0 else '+'} {abs(coef_age):,.0f}·Age "
            f"{'−' if coef_dist < 0 else '+'} {abs(coef_dist):,.0f}·Distance<br>"
            f"<sup>R² = {r2:.4f} &nbsp;|&nbsp; RMSE = ${rmse:,.0f}</sup>"
        ),
        x=0.5, xanchor="center",
        font=dict(size=15),
    ),
    scene=dict(
        xaxis=dict(title="Property Age (years)", backgroundcolor="rgba(240,245,255,0.6)"),
        yaxis=dict(title="Distance to Tech Hub (km)", backgroundcolor="rgba(240,245,255,0.6)"),
        zaxis=dict(title="Sale Price ($)", backgroundcolor="rgba(248,248,248,0.6)"),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9)),
        aspectratio=dict(x=1, y=1, z=0.75),
    ),
    legend=dict(
        x=0.01, y=0.98,
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="#ccc", borderwidth=1,
    ),
    margin=dict(l=0, r=0, t=90, b=0),
    width=1050,
    height=720,
    paper_bgcolor="white",
)

fig.write_html("ols_3d_regression.html")
print("\n✓  Plot saved → ols_3d_regression.html")
fig.show()

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.876
Model:                            OLS   Adj. R-squared:                  0.875
Method:                 Least Squares   F-statistic:                     868.6
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.77e-112
Time:                        20:11:16   Log-Likelihood:                -2935.3
No. Observations:                 250   AIC:                             5877.
Df Residuals:                     247   BIC:                             5887.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 7.932e+05 